In [8]:
# Cell 1: Imports
from FairnessPP_utils import FairnessPredictor, ModelConfig, load_chicago_data
import pandas as pd

In [9]:
# Cell 2: API Component 1 - Configuration
# Demonstrating the "Contract" layer. We can define config without running logic.
config = ModelConfig(n_estimators=10, max_iter_mitigation=5)
print("Configuration Object Created:")
print(config)

Configuration Object Created:
ModelConfig(n_estimators=10, random_state=42, max_iter_mitigation=5)


In [10]:
# Cell 3: API Component 2 - The Wrapper Class
# Initialize the wrapper
predictor = FairnessPredictor(config=config)
print(f"\nWrapper Initialized: {type(predictor)}")
print(f"Default Mitigation State: {predictor.is_mitigated}")


Wrapper Initialized: <class 'FairnessPP_utils.FairnessPredictor'>
Default Mitigation State: False


In [11]:
# Cell 4: Quick Data Load (Small subset for API demo)
# We use the utility loader but just take the first 100 rows to demonstrate the API
X, y, A, _ = load_chicago_data()
X_demo, y_demo, A_demo = X.head(100), y.head(100), A.head(100)
print(f"\nDemo Data Loaded: {X_demo.shape}")

Loading cached data from data/chicago_crime_2020_2023.csv...

Demo Data Loaded: (100, 3)


In [12]:
# Cell 5: API Usage - Baseline Training
# Demonstrating the .train() interface with mitigate=False
predictor.train(X_demo, y_demo, mitigate=False)
print("Baseline training complete.")

Training Baseline Model (Native GradientBoosting)...
Baseline training complete.


In [13]:
# Cell 6: API Usage - Mitigated Training
# Demonstrating the .train() interface with mitigate=True
# This confirms the wrapper correctly switches to Fairlearn internally
predictor.train(X_demo, y_demo, A=A_demo, mitigate=True)
print("Mitigated training complete.")

Training Mitigated Model (Fairlearn ExponentiatedGradient)...


Mitigated training complete.


In [14]:
# Cell 7: API Usage - Evaluation Contract
# Demonstrating that .evaluate() returns our structured dataclass, not just a dict
result = predictor.evaluate(X_demo, y_demo, A_demo)
print("\nEvaluation Object Type:", type(result))
print("Accuracy:", result.accuracy)
print("Fairness Disparity:", result.fairness_disparity)
print("\nGroup Metrics Table Head:")
print(result.group_metrics.head())


Evaluation Object Type: <class 'FairnessPP_utils.EvaluationResult'>
Accuracy: 0.74
Fairness Disparity: 1.0

Group Metrics Table Head:
                      Selection Rate  Accuracy
Intersectional_Group                          
Black_High                  0.133333  0.666667
Black_Low                   0.156250  0.812500
Hispanic_High               0.181818  0.545455
Hispanic_Low                0.000000  0.944444
White_High                  0.250000  0.875000
